# Claude Agent SDK 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台，使用 **Claude Agent SDK** (Python) 构建 AI Agent 应用。

## 功能简介

Claude Agent SDK 是 Anthropic 官方推出的 Agent 开发工具包，可以让开发者快速构建具备工具调用、多轮对话等能力的 AI Agent。

- **一次性查询（One-shot Query）**：使用 `query()` 函数进行单次 Agent 交互，返回异步消息流
- **自定义工具（Custom Tools）**：通过 `@tool` 装饰器定义自定义工具，配合 MCP Server 让 Agent 调用
- **多轮对话（Multi-turn Conversation）**：使用 `ClaudeSDKClient` 进行有状态的多轮会话

## 1. 环境准备

安装 `claude-agent-sdk` 依赖包，并设置环境变量指向七牛 AIToken 平台。

> **前置条件**：需要 Python 3.10 或更高版本。

In [ ]:
# 安装 Claude Agent SDK
%pip install claude-agent-sdk -q

In [ ]:
import os
from claude_agent_sdk import (
    query,
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
)

# 七牛 AIToken 平台地址（Anthropic Messages API 兼容）
os.environ["ANTHROPIC_BASE_URL"] = "https://api.qnaigc.com"

# 从环境变量读取 API Key
api_key = os.environ.get("QINIU_API_KEY", "<your-api-key>")
os.environ["ANTHROPIC_API_KEY"] = api_key

# 设置模型
os.environ["ANTHROPIC_MODEL"] = "claude-4.6-sonnet"

print("环境配置完成!")
print(f"  API 端点: {os.environ['ANTHROPIC_BASE_URL']}")
print(f"  模型: {os.environ['ANTHROPIC_MODEL']}")

## 2. 示例一：一次性查询（One-shot Query）

使用 `query()` 函数进行最简单的 Agent 交互。`query()` 返回一个异步迭代器，可以流式接收 Agent 的响应消息。

消息类型说明：
- `AssistantMessage`：Agent 的回复，包含 `TextBlock`（文本）和 `ToolUseBlock`（工具调用）
- `ResultMessage`：最终结果，包含耗时和费用信息

In [ ]:
# 简单的一次性查询
async def simple_query():
    async for message in query(prompt="请用一句话介绍你自己。"):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"Claude: {block.text}")
        elif isinstance(message, ResultMessage):
            print(f"\n--- 执行信息 ---")
            print(f"耗时: {message.duration_ms}ms")
            print(f"费用: ${message.total_cost_usd:.4f}")

await simple_query()

### 带配置的查询

通过 `ClaudeAgentOptions` 可以定制 Agent 行为，包括系统提示词、最大轮次、可用工具等。

In [ ]:
# 带自定义配置的查询
async def query_with_options():
    options = ClaudeAgentOptions(
        system_prompt="你是一位专业的中国古诗词鉴赏家，请用优美的中文回答问题。",
        max_turns=1,
    )

    async for message in query(
        prompt="请推荐一首描写春天的唐诗，并简要赏析。",
        options=options,
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)

await query_with_options()

## 3. 示例二：自定义工具（Custom Tools）

Claude Agent SDK 支持通过 `@tool` 装饰器定义自定义工具，然后使用 `create_sdk_mcp_server()` 将工具注册为一个 MCP（Model Context Protocol）Server。

Agent 在执行任务时，可以自动判断何时需要调用这些工具，实现"能力扩展"。

### 工具定义三要素
1. **工具名称**：唯一标识，如 `"add"`
2. **工具描述**：让 Agent 理解工具的用途
3. **参数 Schema**：定义输入参数的类型

In [ ]:
from typing import Any
from datetime import datetime
from claude_agent_sdk import tool, create_sdk_mcp_server


# 定义自定义工具：加法
@tool("add", "将两个数字相加", {"a": float, "b": float})
async def add_numbers(args: dict[str, Any]) -> dict[str, Any]:
    result = args["a"] + args["b"]
    return {
        "content": [{"type": "text", "text": f"{args['a']} + {args['b']} = {result}"}]
    }


# 定义自定义工具：乘法
@tool("multiply", "将两个数字相乘", {"a": float, "b": float})
async def multiply_numbers(args: dict[str, Any]) -> dict[str, Any]:
    result = args["a"] * args["b"]
    return {
        "content": [{"type": "text", "text": f"{args['a']} * {args['b']} = {result}"}]
    }


# 定义自定义工具：获取当前时间
@tool("get_current_time", "获取当前日期和时间", {})
async def get_current_time(args: dict[str, Any]) -> dict[str, Any]:
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return {
        "content": [{"type": "text", "text": f"Current time: {now}"}]
    }


print("自定义工具定义完成: add, multiply, get_current_time")

In [ ]:
# 创建 MCP Server，注册所有自定义工具
calculator_server = create_sdk_mcp_server(
    name="calculator",
    version="1.0.0",
    tools=[add_numbers, multiply_numbers, get_current_time],
)

# 配置 Agent 使用自定义工具
tool_options = ClaudeAgentOptions(
    mcp_servers={"calc": calculator_server},
    allowed_tools=[
        "mcp__calc__add",
        "mcp__calc__multiply",
        "mcp__calc__get_current_time",
    ],
    system_prompt="你是一个智能计算助手。当用户要求计算时，请使用提供的工具。",
)


# 使用自定义工具进行查询
async def query_with_tools():
    async for message in query(
        prompt="请计算 15 + 27，然后将结果乘以 3。最后告诉我现在的时间。",
        options=tool_options,
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"Claude: {block.text}")
                elif isinstance(block, ToolUseBlock):
                    print(f"[工具调用] {block.name}，参数: {block.input}")


await query_with_tools()

## 4. 示例三：多轮对话（Multi-turn Conversation）

使用 `ClaudeSDKClient` 可以进行有状态的多轮对话。Client 在多次 `query()` 调用之间保持对话上下文，Agent 可以引用之前的对话内容。

关键特性：
- **会话保持**：多轮对话共享上下文
- **异步生命周期**：通过 `async with` 管理 Client 的启动和关闭
- **响应驱动**：接收响应后可以根据内容发起后续查询

In [ ]:
# 多轮对话示例
async def multi_turn_conversation():
    options = ClaudeAgentOptions(
        system_prompt="你是一位旅行顾问，擅长为用户推荐旅行目的地和规划行程。",
        max_turns=3,
    )

    async with ClaudeSDKClient(options=options) as client:
        # 第一轮：提出需求
        print("=== 第一轮对话 ===")
        await client.query("我想在五月份去一个适合拍照的地方旅行，有什么推荐？")

        async for msg in client.receive_response():
            if isinstance(msg, AssistantMessage):
                for block in msg.content:
                    if isinstance(block, TextBlock):
                        print(f"Claude: {block.text}")
            elif isinstance(msg, ResultMessage):
                print(f"(耗时: {msg.duration_ms}ms)")

        # 第二轮：追问细节（Agent 会记住上一轮推荐的内容）
        print("\n=== 第二轮对话 ===")
        await client.query("你推荐的第一个地方，帮我规划一个3天的行程吧。")

        async for msg in client.receive_response():
            if isinstance(msg, AssistantMessage):
                for block in msg.content:
                    if isinstance(block, TextBlock):
                        print(f"Claude: {block.text}")
            elif isinstance(msg, ResultMessage):
                print(f"(耗时: {msg.duration_ms}ms)")


await multi_turn_conversation()

## 5. 示例四：多轮对话 + 工具调用

结合 `ClaudeSDKClient` 和自定义工具，实现一个具备"记忆"和"能力"的 Agent。

In [ ]:
# 多轮对话 + 工具调用
async def multi_turn_with_tools():
    # 复用之前定义的 calculator_server
    options = ClaudeAgentOptions(
        mcp_servers={"calc": calculator_server},
        allowed_tools=[
            "mcp__calc__add",
            "mcp__calc__multiply",
        ],
        system_prompt="你是一个数学辅导老师，善于用简单的语言解释数学概念。需要计算时请使用工具。",
    )

    async with ClaudeSDKClient(options=options) as client:
        # 第一轮：提出问题
        print("=== 第一轮 ===")
        await client.query("一个长方形的长是 12.5，宽是 8，它的面积是多少？")

        async for msg in client.receive_response():
            if isinstance(msg, AssistantMessage):
                for block in msg.content:
                    if isinstance(block, TextBlock):
                        print(f"Claude: {block.text}")
                    elif isinstance(block, ToolUseBlock):
                        print(f"[计算] {block.name}: {block.input}")

        # 第二轮：基于上一轮继续
        print("\n=== 第二轮 ===")
        await client.query("如果长增加一倍，新的面积是多少？比之前大了多少？")

        async for msg in client.receive_response():
            if isinstance(msg, AssistantMessage):
                for block in msg.content:
                    if isinstance(block, TextBlock):
                        print(f"Claude: {block.text}")
                    elif isinstance(block, ToolUseBlock):
                        print(f"[计算] {block.name}: {block.input}")


await multi_turn_with_tools()

## 6. 参数说明

### 环境变量配置

| 环境变量 | 说明 |
|----------|------|
| `ANTHROPIC_BASE_URL` | API 端点地址，设为 `https://api.qnaigc.com` 指向七牛 AIToken 平台 |
| `ANTHROPIC_API_KEY` | API Key，使用七牛 AIToken 的 API Key |
| `ANTHROPIC_MODEL` | 模型 ID，如 `claude-4.6-sonnet` |

### ClaudeAgentOptions 主要参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `system_prompt` | string | 系统提示词，定义 Agent 角色和行为 |
| `max_turns` | integer | 最大交互轮次 |
| `allowed_tools` | list[str] | 允许 Agent 使用的工具列表 |
| `disallowed_tools` | list[str] | 禁止 Agent 使用的工具列表 |
| `mcp_servers` | dict | MCP Server 配置，键为服务器名，值为 Server 对象 |
| `permission_mode` | string | 权限模式：`default`、`acceptEdits`、`plan`、`bypassPermissions` |
| `model` | string | 模型名称或别名 |
| `env` | dict | 传递给 Agent 子进程的环境变量 |
| `cwd` | Path | 工作目录 |

### @tool 装饰器

```python
@tool(name: str, description: str, schema: dict)
async def my_tool(args: dict[str, Any]) -> dict[str, Any]:
    return {"content": [{"type": "text", "text": "result"}]}
```

| 参数 | 类型 | 说明 |
|------|------|------|
| `name` | string | 工具名称，需唯一 |
| `description` | string | 工具描述，帮助 Agent 理解何时使用 |
| `schema` | dict | 参数 Schema，如 `{"a": float, "b": float}` |

### 工具命名约定

注册到 MCP Server 后的工具名称格式为：`mcp__<server_name>__<tool_name>`

例如：服务器名 `calc`，工具名 `add` → `mcp__calc__add`

### 消息类型

| 类型 | 说明 |
|------|------|
| `AssistantMessage` | Agent 回复，`content` 包含 `TextBlock` 和/或 `ToolUseBlock` |
| `ResultMessage` | 执行完成，包含 `total_cost_usd`（费用）和 `duration_ms`（耗时） |
| `TextBlock` | 文本内容块，通过 `.text` 获取文本 |
| `ToolUseBlock` | 工具调用块，包含 `.name`（工具名）和 `.input`（参数） |

## 7. 注意事项

1. **异步运行**：Claude Agent SDK 基于 `asyncio`，在 Jupyter Notebook 中可直接使用 `await`，在普通 Python 脚本中需要使用 `asyncio.run()`
2. **环境变量方式 vs env 参数**：本示例通过 `os.environ` 设置环境变量。也可以通过 `ClaudeAgentOptions(env={...})` 传递，效果相同
3. **工具权限**：使用自定义工具时，必须在 `allowed_tools` 中明确列出允许的工具名称
4. **模型兼容性**：七牛 AIToken 平台支持的模型请参考平台文档，当前示例使用 `claude-4.6-sonnet`

## 8. 替代配置方式

除了通过 `os.environ` 设置环境变量，也可以通过 `ClaudeAgentOptions` 的 `env` 参数直接传递：

In [ ]:
# 替代方式：通过 ClaudeAgentOptions 的 env 参数配置
env_options = ClaudeAgentOptions(
    env={
        "ANTHROPIC_BASE_URL": "https://api.qnaigc.com",
        "ANTHROPIC_API_KEY": os.environ.get("QINIU_API_KEY", "<your-api-key>"),
        "ANTHROPIC_MODEL": "claude-4.6-sonnet",
    },
    system_prompt="你好，请用中文回答。",
    max_turns=1,
)


# 使用上述配置进行查询
async def query_with_env_options():
    async for message in query(prompt="你好，请简单自我介绍。", options=env_options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)


await query_with_env_options()

## 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [Claude Agent SDK Python 文档](https://github.com/anthropics/claude-agent-sdk-python)
- [MCP (Model Context Protocol) 文档](https://modelcontextprotocol.io)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com)